# Agentic AI - Fuel Forecasting

In [ ]:
# Installation of packages

!pip install crewai langchain-google-genai

In [ ]:
# import codes

import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM
#from crewai_tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.3
)

llm.invoke("Say hello")

In [ ]:
# 1. Setup API Key and LLM
# IMPORTANT: Replace "YOUR_API_KEY" with your actual Google Gemini API Key
# Ensure you have obtained a key from Google AI Studio.

os.environ["GOOGLE_API_KEY"] = ""
# Set a dummy OpenAI API key to prevent crewai from failing when it internally tries to initialize an OpenAI client.
#os.environ["OPENAI_API_KEY"] = "sk-dummy"
gemini_llm = ChatGoogleGenerativeAI(model="gemini-pro")

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# Define state structure
class MyState(TypedDict):
    input: str
    output: str

In [ ]:
# Node function
def chatbot_node(state: MyState):
    response = llm.invoke(state["input"])
    return {"output": response.content}

# Build graph
graph = StateGraph(MyState)
graph.add_node("chatbot", chatbot_node)

graph.set_entry_point("chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()

In [ ]:
result = app.invoke({"input": "Brent oil price forecast"})
print(result["output"])


# AGENT CREATION FOR BRENT CRUDE FORECASTING

In [ ]:
!pip install -q duckduckgo-search

In [ ]:

# ============================================
# IMPORTS
# ============================================
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
import yfinance as yf
from duckduckgo_search import DDGS

# ============================================
# INITIALIZE LLM
# ============================================
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

In [69]:

# Safe text extraction
def extract_text(response):
    if hasattr(response, "content"):
        content = response.content
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            text_parts = []
            for part in content:
                if isinstance(part, dict) and "text" in part:
                    text_parts.append(part["text"])
                else:
                    text_parts.append(str(part))
            return "".join(text_parts)
    return str(response)

# ============================================
# STATE
# ============================================
class StockState(TypedDict):
    ticker: str
    price_data: str
    news: List[str]
    analysis: str
    final_report: str

# ============================================
# NODE 1 — FETCH STOCK DATA - Crude Oil WTI
# ============================================
def fetch_stock_data(state: StockState):
    ticker = state["ticker"]
    stock = yf.Ticker(ticker)
    hist = stock.history(period="5d")

    if hist.empty:
        return {"price_data": "No data found"}

    latest_close = hist["Close"].iloc[-1]
    change = latest_close - hist["Close"].iloc[-2]

    summary = f"""
Ticker: {ticker}
Latest Close Price: {latest_close:.2f}
1-Day Change: {change:.2f}
5-Day Trend:
{hist['Close'].to_string()}
"""
    return {"price_data": summary}

# ============================================
# NODE 2 — FETCH NEWS (DuckDuckGo)
# ============================================
def fetch_news(state: StockState):
    ticker = state["ticker"]
    news_list = []

    with DDGS() as ddgs:
        results = ddgs.news(f"{ticker} stock news", max_results=5)
        for r in results:
            news_list.append(r["title"])

    return {"news": news_list}

# ============================================
# NODE 3 — ANALYZE WITH GEMINI
# ============================================
def analyze(state: StockState):
    prompt = f"""
You are a financial analyst.

Stock Data:
{state['price_data']}

Recent News Headlines:
{state['news']}

Provide:
1. Technical trend insight
2. Sentiment from news
3. Short-term outlook
4. Risk level (Low/Medium/High)
"""
    response = llm.invoke(prompt)
    text = extract_text(response)

    return {"analysis": text}

# ============================================
# NODE 4 — FINAL REPORT FORMAT
# ============================================
def finalize(state: StockState):
    final_prompt = f"""
Create a professional stock analysis report:

Ticker: {state['ticker']}

{state['analysis']}
"""
    response = llm.invoke(final_prompt)
    text = extract_text(response)

    return {"final_report": text}

# ============================================
# BUILD GRAPH
# ============================================
graph = StateGraph(StockState)

graph.add_node("fetch_stock_data", fetch_stock_data)
graph.add_node("fetch_news", fetch_news)
graph.add_node("analyze", analyze)
graph.add_node("finalize", finalize)

graph.set_entry_point("fetch_stock_data")
graph.add_edge("fetch_stock_data", "fetch_news")
graph.add_edge("fetch_news", "analyze")
graph.add_edge("analyze", "finalize")
graph.add_edge("finalize", END)

app = graph.compile()

# ============================================
# RUN ANALYSIS
# ============================================
result = app.invoke({"ticker": "BZ=F"})

print("\n============================")
print("FINAL STOCK REPORT- Brent Crude Oil Futures (BZ=F)")
print("============================\n")
print(result["final_report"])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


FINAL STOCK REPORT- Bloomberg Brent Crude

## Professional Stock Analysis Report: Brent Crude Oil Futures (BZ=F)

**Date:** March 11, 2024

**Disclaimer:** This report is for informational purposes only and does not constitute financial advice. Investors should conduct their own due diligence and consult with a qualified financial professional before making any investment decisions. Futures trading involves substantial risk of loss and is not suitable for all investors.

---

### **1. Asset Identification & Assumptions**

*   **Ticker:** BZ=F
*   **Assumed Asset:** Based on the ticker BZ=F and the accompanying news headline mentioning "oil tanks," it is assumed that BZ=F represents **Brent Crude Oil futures**. This analysis proceeds with that assumption.

---

### **2. Technical Trend Analysis**

Brent Crude Oil futures (BZ=F) exhibited a strong bullish momentum in the preceding days, climbing from a low of **$85.41** on March 5th to a peak of **$98.96** by March 9th. This represented

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag